# 00 はじめに — 基礎知識と用語から始めよう（推薦システム）

> 🔒 すべて**合成データ**です。⏱ 所要時間: 読む 30分 ＋ 実行 数秒。**難しければ、「図の見方」と「用語集」だけ読んで、次のNotebookへ進んでください。**

予備知識ゼロから始めます。数式はほとんど使いません。分からない言葉が出たら、ここに戻ってきてください。

## 1. このプロジェクトは何をするの？（ひとことで）
**「この人は、どの商品（映画・本・動画）を好きそうか」を当てて、おすすめを出す仕組み**（推薦システム）を、自分の手で作ります。動画サイトの「あなたへのおすすめ」、ネット通販の「この商品を買った人は…」と同じ仕組みです。

もう1つの大きなテーマは、**「おすすめの良し悪しを、どう測るか」が意外に難しい**ことです。世の中のデータは、人気のものに偏っているため、**測り方しだいで結論が逆転**します。それを、答えを知っている合成データで確かめます。

## 2. 基礎知識

### 2-1. 評価データの形 — 「ユーザー × 商品」の表
ユーザーが商品に付けた評価（星1〜5）を、表にします。行がユーザー、列が商品。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]

import pandas as pd
R = pd.DataFrame([[5, 4, np.nan, 1, np.nan],
                  [np.nan, 5, 4, np.nan, 1],
                  [1, np.nan, 2, 5, 4],
                  [np.nan, 1, np.nan, 4, 5]],
                 index=["花子", "太郎", "次郎", "美咲"], columns=["恋愛映画A", "恋愛映画B", "家族映画C", "アクションD", "アクションE"])
R

**出力の見方**: 数字は星の数。**空欄（NaN）は「評価していない」**です。ほとんどのマスが空欄なのが普通（実際のサービスでは、9割以上が空欄）。
ここで大事な注意: **空欄は「嫌い」ではなく「知らない・見ていない」** です。推薦とは、この空欄を埋める（好きそうな商品を見つける）作業です。
この表を見ると、花子さんと太郎さんは「恋愛映画が好き」、次郎さんと美咲さんは「アクションが好き」という**傾向（好みのグループ）** が読み取れます。推薦の手法は、この傾向を自動で見つけます。

### 2-2. 一番かんたんなおすすめ — 人気順
**評価が多い順**に、全員に同じ商品を薦める方法です。意外に強い基準です（みんなが見ているものは、多くの人が好きだから）。
弱点: **全員に同じものを薦める**（個人の好みを無視）。人気がない良い商品は、いつまでも薦められない。

In [ ]:
counts = R.notna().sum()
print("評価された回数:\n", counts.to_string())
print("\n人気順のおすすめ:", list(counts.sort_values(ascending=False).index))

**出力の見方**: 評価の回数が多い順に並べたものが「人気順」です。誰に対しても同じ順番になります。

### 2-3. おすすめの良し悪しを測る — precision@10
おすすめリスト（上位10件）のうち、**本当に好きだったものの割合**を **precision@10** と呼びます。
例: 10件薦めて、そのうち3件が「好き」なら precision@10 = 0.3。**大きいほど良い**。
関連の指標: **NDCG**（好きなものが**上位にあるほど**高得点）、**recall**（好きなもの全体のうち、何割を薦められたか）。

In [ ]:
recommended = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J"]    # おすすめ上位10件（例）
liked = {"B", "D", "J", "Z"}                                         # 本当に好きなもの（例）
hits = [r in liked for r in recommended]
print("好きだったか:", hits)
print("precision@10 =", sum(hits) / len(recommended))
print("recall@10 =", sum(hits) / len(liked), "（好きな4件のうち、3件を薦められた）")

**出力の見方**: 上位10件のうち、B・D・J の3件が当たり。precision@10 = 3/10 = 0.3。好きなものは全部で4件(B, D, J, Z)で、そのうち3件を薦められたので recall = 0.75。

### 2-4. 似た人・似た商品から薦める — 近傍法
**「あなたと似た人が好きなものを薦める」** 方法です。似ているかどうかは、評価のパターンの**重なり具合（コサイン類似度）**で測ります。

In [ ]:
M = R.fillna(0).to_numpy()
def cos(a, b): return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
names = list(R.index)
print("花子さんと他の人の似ている度（1に近いほど似ている）")
for i in range(1, 4):
    print(f"  {names[i]}: {cos(M[0], M[i]):.2f}")

**出力の見方**: 花子さんに最も似ているのは太郎さん（恋愛映画好き同士）。次郎さん・美咲さんはアクション好きなので似ていません。だから、花子さんには太郎さんが高く評価した「家族映画C」が薦められます。

### 2-5. 好みを少数の「軸」で表す — 行列分解
もっと賢い方法です。ユーザーと商品を、**少数の軸（例: 恋愛度・アクション度）** の数字で表します。
- ユーザーの数字 × 商品の数字 を掛けて足すと、**「好みの合いそう度」** になる。
- 評価の表を、「ユーザー表 × 商品表」という**2つの小さな表の掛け算**に分解する → **行列分解**。軸（**潜在因子**）は、データから自動で見つかります。

In [ ]:
users = np.array([[0.9, 0.1], [0.8, 0.2], [0.1, 0.9], [0.2, 0.8]])        # 各ユーザーの「恋愛度, アクション度」
items = np.array([[5, 0.5], [4.5, 0.5], [3, 1.5], [0.5, 5], [0.5, 4.5]])    # 各商品の「恋愛らしさ, アクションらしさ」
pred = users @ items.T                                                         # 掛け算＝好みの合いそう度
print(np.round(pred, 1))

**出力の見方**: 4×5の表は、各ユーザーと各商品の「合いそう度」です。最初の2行（恋愛好き）は前半の商品が高く、後半の2行（アクション好き）は後半の商品が高い。**少数の軸だけで、元の表のパターンを再現できる**のが、行列分解の強みです。

### 2-6. 偏りという落とし穴 — 人気に偏る
実際のデータは、**人気の商品に評価が集中**します。

In [ ]:
rng = np.random.default_rng(0)
pop = np.exp(1.2 * rng.normal(size=500))                      # 商品の人気（少数が非常に人気）
cnt = np.sort(rng.multinomial(25000, pop / pop.sum()))[::-1]  # 評価の回数
plt.figure(figsize=(8, 3.2)); plt.bar(range(500), cnt, width=1.0)
plt.xlabel("商品（評価の多い順）"); plt.ylabel("評価された回数"); plt.grid(alpha=0.3)
share = cnt[:50].sum() / cnt.sum(); plt.title(f"上位10%の商品が、全評価の {share:.0%} を占める"); plt.show()

**図の見方**: 左端の少数の商品に、評価が山のように集まり、右の大多数の商品はほとんど評価されません（**ロングテール**）。
これが問題になる理由: **評価データだけで「おすすめの良し悪し」を測ると、人気の商品を薦める手法が、実力以上に良く見える**のです。「人気 ＝ みんなが見ている ＝ テストデータにも出てくる」から、当たりやすい。しかし、それは**個人の好みに合わせた**おすすめではありません。

このプロジェクトでは、合成データを使い、**観測できる評価（偏りあり）** と **本当の好み（答え）** の両方で評価して、このずれを確かめます。

### 2-7. 新しいユーザー・新しい商品 — コールドスタート
評価が1つもない新しい人・商品には、上の方法が使えません。**商品のジャンルなどの特徴**を使う方法（コンテンツベース）や、評価が集まるまでの工夫が必要です。

### 2-8. 推薦を繰り返すと — フィードバックループ
おすすめした商品が見られて、評価が増え、さらに薦められる…という循環で、**人気の偏りが増幅**することがあります。

## 3. 用語集（ここに戻ってきてください）

| 用語 | やさしい説明 |
|---|---|
| **推薦システム** | ユーザーに、好みそうな商品を薦める仕組み。 |
| **評価（レーティング）** | ユーザーが商品に付けた点数（星1〜5）。 |
| **関連あり（好き）** | ここでは、評価が4以上、または本当の好みが4以上のこと。 |
| **人気順** | 評価の多い順に、全員に同じものを薦める基準。 |
| **平均評価順** | 商品の平均点が高い順に薦める基準（これも全員に同じ）。 |
| **個人化** | 人ごとに違う商品を薦めること。推薦の本来の目的。 |
| **precision@10 / recall@10 / NDCG@10** | 上位10件の当たりの割合／好きなもののうち何割を薦められたか／上位ほど高得点にした指標。 |
| **カバレッジ** | 薦められた商品の種類の割合。低いと、全員に同じ少数を薦めている。 |
| **露出のジニ係数** | 薦める回数の偏り（0=均等、1=1つに集中）。 |
| **近傍法（kNN）** | 似た人・似た商品の評価から薦める。 |
| **行列分解（ALS・BPR）** | ユーザーと商品を少数の軸で表す。ALSは点数を当て、BPRは順位を学ぶ。 |
| **潜在因子** | データから自動で見つかる、好みの軸。 |
| **観測テスト** | 実際に評価されたデータで測るテスト（偏りあり）。 |
| **本当の好み** | 合成データだけが知っている答え（偏りなし）。現実には手に入らない。 |
| **合成データ** | プログラムで作った練習用データ。答えが分かる。 |
| **MNAR** | 評価されるかどうかが、好みそのものに左右されること（好きなものほど評価される）。 |
| **逆確率重み付け（IPS/IPW）** | 評価されにくかったものを重く扱って、偏りを補正する方法。 |
| **コールドスタート** | 評価が無い・少ない新しい人／商品に薦める問題。 |
| **フィードバックループ** | 薦める→見られる→さらに薦められる、という循環。 |
| **標準誤差** | 平均のあやふやさ。差がその2倍より小さいときは、差があるとは言えない。 |
| **過学習** | 学習データには合うが、新しいデータには合わないこと。 |

## 4. このプロジェクトの地図（読む順）

| Notebook | 問い | 一言 |
|---|---|---|
| 00 データと基準 | 簡単なおすすめを、どう測る？ | 観測と本当で、順位が逆転する |
| 01 評価の落とし穴 | 測り方で結論が変わる？ | 変わる |
| 02 近傍法 | 似た人から薦めると？ | 入力しだいで人気順と同じになる |
| 03 行列分解 | 評価の点数を使うと？ | 個人化が初めて効く |
| 04 コールドスタート | 新商品には？ | ジャンルを使う |
| 05 偏りの補正とループ | 補正できる？ 繰り返すと？ | 補正は万能ではない |

## 5. 結果を読むときの心得（5つ）
1. **2種類の評価（観測テストと本当の好み）を両方見る。** 一致しないことがある。
2. **基準（人気順など）と比べる。**
3. **精度だけでなく、カバレッジ（種類の幅）も見る。**
4. **標準誤差と並べる。** 差が小さければ「同じ」。
5. **合成データの結論を、そのまま現実に持ち込まない。**

## 6. よくある疑問
- **Q. 評価していない商品は、嫌いなの？** A. いいえ。知らないだけです。そこが推薦の難しさです。
- **Q. 人気順が強いなら、それで十分では？** A. 全員に同じ物を薦めるので、個人の好みに合わせたおすすめにはなりません。
- **Q. なぜ合成データ？** A. 「本当の好み」の答えを知っているので、おすすめが**本当に良かったか**を確かめられます。

次は **00_data_and_baselines** へ進みましょう。